![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Building Agents with Claude Agent SDK and RedisVL MCP

In this notebook, we'll demonstrate how to build an agent using Claude Agent SDK and give it access to the RedisVL MCP Toolkit.

## Before we get started
To understand more about the frameworks used in this notebook, refer to the following resources:
- [RedisVL MCP](https://github.com/redis/redis-vl-python#mcp-server) - GitHub repository to the RedisVL Project. We are using the MCP functionality in this notebook.
- [Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk/) - Documentation for Claude Agent SDK. We will be using it as the base framework to build our agent.

## Let's Begin!
<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/MCP/01_claude_agent_sdk_redisvl_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

First, let's download the required packages and set our API keys:

In [1]:
%pip install -qU "redisvl[mcp,sentence-transformers,nltk]" claude-agent-sdk 


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Install Redis Stack

In this notebook, Redis is used to store, index, and query vector
embeddings created from hypothetical texts. **We need to make sure we have a Redis
instance available.**

#### For Colab
Use the shell script below to download, extract, and install [Redis Stack](https://redis.io/docs/getting-started/install-stack/) directly from the Redis package archive.

In [ ]:
# NBVAL_SKIP
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

#### For Alternative Environments
There are many ways to get the necessary redis-stack instance running
1. On cloud, deploy a [FREE instance of Redis in the cloud](https://redis.com/try-free/). Or, if you have your
own version of Redis Enterprise running, that works too!
2. Per OS, [see the docs](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/)
3. With docker: `docker run -d --name redis-stack-server -p 6379:6379 redis/redis-stack-server:latest`

### Define the Redis Connection URL

By default this notebook connects to the local instance of Redis Stack. **If you have your own Redis Enterprise instance** - replace REDIS_PASSWORD, REDIS_HOST and REDIS_PORT values with your own.

In [2]:
import os

# Replace values below with your own if using Redis Cloud instance
REDIS_HOST = os.getenv("REDIS_HOST", "localhost") # ex: "redis-18374.c253.us-central1-1.gce.cloud.redislabs.com"
REDIS_PORT = os.getenv("REDIS_PORT", "6379")      # ex: 18374
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")  # ex: "1TNxTEdYRDgIDKM2gDfasupCADXXXX"

# If SSL is enabled on the endpoint, use rediss:// as the URL prefix
REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"
os.environ["REDIS_URL"] = REDIS_URL

## Imports

In [3]:
import json
from pprint import pprint
import getpass
import warnings
import yaml

import numpy as np

warnings.filterwarnings("ignore")

## API keys

In [ ]:
# NBVAL_SKIP
# Anthropic API key is required for agent calls
os.environ['ANTHROPIC_API_KEY'] = os.getenv("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key: ")

## Setting up Redis

### Sample Data

The sample data contains a fictional knowledge base that contains support knowledge records stored as structured documents for retrieval and update. It includes runbooks, incident summaries, KB articles, and release notes, along with metadata and text content that can be indexed for search.

We provide our agent with the `search-records` and `upsert-records` tools. Using these tools, the agent should be able to perform the following:
- Retrieval (using `search-records`) - Answer support style questions such as "What should I do after a `eu-central` failover?" or "which KB article covers stale reads?"

- Knowledge Management (using `upsert-records`) - Add a new incident summary or replace an outdated support article after the issue is understood.

In [5]:
with open("resources/knowledge_records.json", "r") as f:
    knowledge_records = json.load(f)

pprint(knowledge_records)

[{'content': 'After a regional failover in eu-central, elevated cache miss '
             'rate usually means replicas are warm but invalidation workers '
             'are behind. Restart the cache warming job, replay invalidation '
             'events, and pre-warm the top keys for the developer platform '
             'before reopening traffic.',
  'last_reviewed_at': '2026-03-18',
  'product': 'developer-platform',
  'record_id': 'runbook_cache_failover_eu_central',
  'region': 'eu-central',
  'release_version': 'na',
  'severity': 'sev1',
  'source_type': 'runbook',
  'team': 'platform',
  'title': 'Runbook: mitigate elevated cache miss rate after eu-central '
           'failover'},
 {'content': 'A March 2026 failover in eu-central caused elevated cache miss '
             'rate and stale responses for the developer portal. The fix was '
             'to drain the legacy invalidation backlog, force a cache warmup '
             'pass, and keep the platform severity at sev1 until

### Creating the Redis index

A Redis index defines how Redis should store and search our data, including the vector embeddings used for similarity search.

We populate it with sample data that we will retrieve using our agents and the tools exposed by the MCP server.


In [6]:
# Vectorizer for creating document embeddings
from redisvl.utils.vectorize import HFTextVectorizer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
vectorizer = HFTextVectorizer(model=EMBEDDING_MODEL)

#### Define the index

Using a schema, we define an index called `knowledge`, the collection of fields to be indexed and how they are indexed.

In [7]:
INDEX_NAME = "knowledge"

knowledge_schema = {
    "index": {
        "name": INDEX_NAME,
        "prefix": "knowledge",
        "storage_type": "hash",
    },
    "fields": [
        {"name": "record_id", "type": "tag"},
        {"name": "title", "type": "text"},
        {"name": "content", "type": "text"},
        {"name": "source_type", "type": "tag"},
        {"name": "team", "type": "tag"},
        {"name": "region", "type": "tag"},
        {"name": "product", "type": "tag"},
        {"name": "severity", "type": "tag"},
        {"name": "release_version", "type": "tag"},
        {"name": "last_reviewed_at", "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": vectorizer.dims,
                "distance_metric": "cosine",
                "algorithm": "hnsw",
                "datatype": "float32",
            },
        },
    ],
}
pprint(knowledge_schema)

{'fields': [{'name': 'record_id', 'type': 'tag'},
            {'name': 'title', 'type': 'text'},
            {'name': 'content', 'type': 'text'},
            {'name': 'source_type', 'type': 'tag'},
            {'name': 'team', 'type': 'tag'},
            {'name': 'region', 'type': 'tag'},
            {'name': 'product', 'type': 'tag'},
            {'name': 'severity', 'type': 'tag'},
            {'name': 'release_version', 'type': 'tag'},
            {'name': 'last_reviewed_at', 'type': 'text'},
            {'attrs': {'algorithm': 'hnsw',
                       'datatype': 'float32',
                       'dims': 384,
                       'distance_metric': 'cosine'},
             'name': 'embedding',
             'type': 'vector'}],
 'index': {'name': 'knowledge', 'prefix': 'knowledge', 'storage_type': 'hash'}}


#### Create the search index

Create the knowledge index within Redis. It will be queried by the agent via the MCP tools

In [8]:
from redisvl.index import SearchIndex

index = SearchIndex.from_dict(knowledge_schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

In [9]:
# Get info about the knowledge index
!rvl index info -i knowledge



Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
| knowledge     | HASH          | ['knowledge'] | []            | 0             |
╰───────────────┴───────────────┴───────────────┴───────────────┴───────────────╯
Index Fields:
╭──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────╮
│ Name             │ Attribute        │ Type             │ Field Option     │ Option Value     │ Field Option     │ Option Value     │ Field Option     │ Option Value     │ Field Option     │ Option Value     │ Field Option     │ Option Value     │ Field Option     │ 

### Populating the Redis index

At this stage, we have defined the index containing the documents we need for the agent. Next, we want to populate the index with records from our knowledge base.

In [10]:
# Embed the knowledge records and load them into the index
embeddings = vectorizer.embed_many(
    contents=[record["content"] for record in knowledge_records],
    batch_size=8,
)

records_with_embeddings = [
    {**record, "embedding": np.array(embedding, dtype=np.float32).tobytes()}
    for record, embedding in zip(knowledge_records, embeddings)
]

keys = index.load(records_with_embeddings, id_field="record_id")

print(f"Created '{INDEX_NAME}' with {len(keys)} records.")
print(keys)

Created 'knowledge' with 10 records.
['knowledge:runbook_cache_failover_eu_central', 'knowledge:incident_cache_failover_summary', 'knowledge:kb_legacy_invalidation_troubleshooting', 'knowledge:release_2026_03_legacy_invalidation_deprecation', 'knowledge:runbook_new_invalidation_cutover', 'knowledge:kb_cache_key_version_mismatch', 'knowledge:incident_release_rollout_cache_regression', 'knowledge:release_2026_04_new_invalidation_endpoint', 'knowledge:runbook_checkout_processor_brownout', 'knowledge:kb_identity_token_expiry']


## Setting up the MCP Server

### Why The MCP Layer Matters


The MCP server is the bridge between the agent and the Redis-backed knowledge base. In this notebook, the important tools are `search-records` for retrieval and `upsert-records` for writes.

### MCP Config

The MCP config is a yaml file that defines how an MCP server is run. In this case, it also specifies how the MCP server interacts with the Redis index.

In [11]:
# For readability, we define the config as a dictionary first. But it should be in yaml format.

mcp_config = {
    "server": {
        "redis_url": REDIS_URL,
    },
    "indexes": {
        INDEX_NAME: {
            "redis_name": INDEX_NAME,
            "vectorizer": {
                "class": "HFTextVectorizer",
                "model": EMBEDDING_MODEL,
            },
            "search": {
                "type": "hybrid",
                "params": {"text_scorer": "BM25STD",
                            "stopwords": "english",
                            "combination_method": "LINEAR",
                            "linear_text_weight": 0.3
                }
            },
            "runtime": {
                "text_field_name": "content",
                "vector_field_name": "embedding",
                "default_embed_text_field": "content",
                "default_limit": 10,
                "max_limit": 25,
                "max_upsert_records": 64,
                "skip_embedding_if_present": True,
                "max_result_window": 100,
                "startup_timeout_seconds": 30,
                "request_timeout_seconds": 60,
                "max_concurrency": 16
            },
        }
    },
}

pprint(yaml.safe_dump(mcp_config, sort_keys=False))

('server:\n'
 '  redis_url: redis://:@localhost:6379\n'
 'indexes:\n'
 '  knowledge:\n'
 '    redis_name: knowledge\n'
 '    vectorizer:\n'
 '      class: HFTextVectorizer\n'
 '      model: sentence-transformers/all-MiniLM-L6-v2\n'
 '    search:\n'
 '      type: hybrid\n'
 '      params:\n'
 '        text_scorer: BM25STD\n'
 '        stopwords: english\n'
 '        combination_method: LINEAR\n'
 '        linear_text_weight: 0.3\n'
 '    runtime:\n'
 '      text_field_name: content\n'
 '      vector_field_name: embedding\n'
 '      default_embed_text_field: content\n'
 '      default_limit: 10\n'
 '      max_limit: 25\n'
 '      max_upsert_records: 64\n'
 '      skip_embedding_if_present: true\n'
 '      max_result_window: 100\n'
 '      startup_timeout_seconds: 30\n'
 '      request_timeout_seconds: 60\n'
 '      max_concurrency: 16\n')


In [12]:
# Write the config into a file called "mcp-config.yaml"
MCP_CONFIG_FILEPATH = os.path.join("config", "redisvl-mcp-config.yaml")
os.makedirs(os.path.dirname(MCP_CONFIG_FILEPATH), exist_ok=True)

with open(MCP_CONFIG_FILEPATH, "w", encoding="utf-8") as f:
    f.write(yaml.safe_dump(mcp_config, sort_keys=False))


### Running the MCP server

The direct server checks below are only sanity checks. The real evaluation comes later, when the agent decides on its own which MCP tool to call for each prompt.

To simulate the MCP server running in a separate process and independent of the client, we use `subprocess.Popen`. The MCP server should be running locally (`127.0.0.1`) on port `16740`.

In [13]:
# NBVAL_SKIP
# Start MCP Server
import subprocess

# Start the process
MCP_SERVER_HOSTNAME = "127.0.0.1"
MCP_SERVER_PORT = "16740"
MCP_TRANSPORT = "streamable-http"

# Run a redisvl mcp server on localhost:16740 on streamable-http
mcp_server = subprocess.Popen(["uv", "run", "rvl", "mcp", 
                                "--config", MCP_CONFIG_FILEPATH, 
                                "--transport", MCP_TRANSPORT, 
                                "--host", MCP_SERVER_HOSTNAME, 
                                "--port", MCP_SERVER_PORT], 
                           stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, text=True)

# Verify that the mcp server is running in another process
! ps ax | grep "mcp" | sed 's|/[^ ]*/||g'

39132   ??  S      0:00.05 uv run rvl mcp --config config/redisvl-mcp-config.yaml --transport streamable-http --host 127.0.0.1 --port 16740
39134   ??  U      0:01.12 python rvl mcp --config config/redisvl-mcp-config.yaml --transport streamable-http --host 127.0.0.1 --port 16740
39133 s005  Ss+    0:00.63 zsh -c  ps ax | grep "mcp" | sed 's|/[^ ]*/||g'
39177 s005  R+     0:00.00 grep mcp


After a few seconds, the MCP server should be up and running.

In [ ]:
# NBVAL_SKIP
# Test TCP connection
!nc -z 127.0.0.1 16740

Connection to 127.0.0.1 port 16740 [tcp/*] succeeded!


In [ ]:
# NBVAL_SKIP
# Ping and test initialization with the MCP server
import requests

MCP_SERVER_URL = f"http://{MCP_SERVER_HOSTNAME}:{MCP_SERVER_PORT}/mcp"

# We need to use a session to connect to the server.
session = requests.Session()
session.headers.update({
    "Accept": "application/json, text/event-stream",
})

resp = session.post(
    MCP_SERVER_URL,
    json={
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "notebook", "version": "0.1"},
        },
    },
    timeout=10,
)

print(resp.status_code)
print(resp.text)


200
event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-03-26","capabilities":{"experimental":{},"logging":{},"prompts":{"listChanged":true},"resources":{"subscribe":false,"listChanged":true},"tools":{"listChanged":true},"extensions":{"io.modelcontextprotocol/ui":{}}},"serverInfo":{"name":"redisvl","version":"3.2.4"}}}




In [ ]:
# NBVAL_SKIP
# We need to use the session ID from the initialization for future requests via HTTP
mcp_session_id = resp.headers.get("mcp-session-id")
session.headers.update({
    "mcp-session-id": mcp_session_id,
})

In [ ]:
# NBVAL_SKIP
# List the tools in the MCP server
resp = session.post(
    MCP_SERVER_URL,
    headers={
        "Accept": "application/json, text/event-stream",
    },
    json={
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/list",
        "params": {},
    },
    timeout=10,
)

print(resp.status_code)
print(resp.text)


200
event: message
data: {"jsonrpc":"2.0","id":2,"result":{"tools":[{"name":"search-records","description":"Search records in the configured Redis index.","inputSchema":{"additionalProperties":false,"properties":{"query":{"type":"string"},"limit":{"anyOf":[{"type":"integer"},{"type":"null"}],"default":null},"offset":{"default":0,"type":"integer"},"filter":{"anyOf":[{"type":"string"},{"additionalProperties":true,"type":"object"},{"type":"null"}],"default":null},"return_fields":{"anyOf":[{"items":{"type":"string"},"type":"array"},{"type":"null"}],"default":null}},"required":["query"],"type":"object"},"_meta":{"fastmcp":{"tags":[]}}},{"name":"upsert-records","description":"Upsert records in the configured Redis index.","inputSchema":{"additionalProperties":false,"properties":{"records":{"items":{"additionalProperties":true,"type":"object"},"type":"array"},"id_field":{"anyOf":[{"type":"string"},{"type":"null"}],"default":null},"skip_embedding_if_present":{"anyOf":[{"type":"boolean"},{"type

At this stage, we have successfully set up our RedisVL MCP server and tested our connection to it.

## Building our agent

In [ ]:
# NBVAL_SKIP
# Build an agent using Claude Agent SDK
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

REDISVL_ALLOWED_TOOLS = ["mcp__redisvl__*"] # Allow all tools by redisvl mcp server
CLAUDE_MODEL = "claude-haiku-4-5"

# Configure the Claude Agent
agent_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    cwd=os.getcwd(),
    system_prompt=(
        "You are an internal support copilot for the developer-platform team. "
        "When a user asks about incidents, runbooks, release notes, or support "
        "KB content, use the RedisVL MCP tools first before answering. Prefer "
        "search-records for retrieval. Only use upsert-records when the user "
        "explicitly asks to add or update knowledge records. Cite the record "
        "titles you used in the final answer."
    ),
    mcp_servers={
        "redisvl": {
            "type": "http",
            "url": MCP_SERVER_URL,
            "env": {
                "REDIS_URL": os.environ["REDIS_URL"] or REDIS_URL,
            },
        }
    },
    allowed_tools=REDISVL_ALLOWED_TOOLS,
    max_turns=6,
)

agent = ClaudeSDKClient(options=agent_options)

## Using our Agent

Now that we have set up the following:
- Redis Index: `knowledge`
- RedisVL MCP Server
- Claude Agent

We can use our agent to perform retrieval and addition/update operations to our `knowledge` index.

In [19]:
# Helper functions
from claude_agent_sdk.types import (
    AssistantMessage,
    ResultMessage,
    SystemMessage,
    TextBlock,
    ToolResultBlock,
    ToolUseBlock,
)

def print_agent_message(message):
    """
    Formats the agent output in a more human-readable way.
    """
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"Claude:\n{block.text}\n")
            elif isinstance(block, ToolUseBlock):
                print(f"Tool use -> {block.name}")
                pprint(block.input)
                print()
            elif isinstance(block, ToolResultBlock):
                print(f"Tool result <- {block.tool_use_id}")
                pprint(block.content)
                print()
    elif isinstance(message, ResultMessage):
        print("Result summary:")
        print(message.result)
        # if message.total_cost_usd is not None:
        #     print(f"Cost: ${message.total_cost_usd:.4f}")

### Retrieval with `search-records`

We use a few prompts to guide the agent to perform retrieval on our knowledge base.

In [20]:
SEARCH_PROMPTS = [
    "Search the internal knowledge base for guidance on mitigating elevated cache miss rate after a eu-central failover. Focus on developer-platform material owned by the platform team, and summarize the recommended actions with the record titles you used.",
    "Find release-note and runbook content about the deprecation of the legacy cache invalidation flow for developer-platform. Summarize what changed and cite the relevant titles.",
    "Look up support knowledge for stale reads or invalidation troubleshooting in developer-platform. Prefer KB-style guidance over incident summaries and include the source type in the answer.",
    "Search for cache regression guidance, but exclude unrelated checkout and identity-service material. I only want developer-platform results with titles and source types.",
]

In [ ]:
# NBVAL_SKIP
async with agent as client:
    for i, prompt in enumerate(SEARCH_PROMPTS, start=1):
        print("=" * 80)
        print(f"Task {i}: {prompt}\n")
        await client.query(prompt)
        async for msg in client.receive_response():
            print_agent_message(msg)
        print("=" * 80)


Task 1: Search the internal knowledge base for guidance on mitigating elevated cache miss rate after a eu-central failover. Focus on developer-platform material owned by the platform team, and summarize the recommended actions with the record titles you used.

Tool use -> mcp__redisvl__search-records
{'limit': 10,
 'query': 'mitigating elevated cache miss rate eu-central failover'}

Tool use -> mcp__redisvl__search-records
{'limit': 10,
 'query': 'cache miss rate failover mitigation remediation developer-platform'}

Claude:
Based on the internal knowledge base search, here are the **recommended actions for mitigating elevated cache miss rate after an EU Central failover**:

## Key Mitigation Steps

According to the **"Runbook: mitigate elevated cache miss rate after eu-central failover"**, the issue typically stems from warm replicas but invalidation workers falling behind. The recommended actions are:

1. **Restart the cache warming job** — Trigger a new cache warm-up cycle to rebuild

### Insert/update with `upsert-records`

Similar to `search-records`, we use a few prompts to guide the agent to perform upsert operations on the knowledge base.

In [22]:
UPSERT_PROMPTS = [
    "Create a new knowledge record with `record_id` `incident_ap_southeast_cache_failover_2026_04`, `source_type` `incident_summary`, `team` `platform`, `region` `ap-southeast`, `product` `developer-platform`, `severity` `sev2`, `release_version` `na`, `last_reviewed_at` `2026-04-09`, title `Incident summary: cache warmup lag after ap-southeast failover`, and content `After failover to ap-southeast, cache hit rate dropped because warmup workers started late. The mitigation was to replay invalidation events, pre-warm the highest-traffic keys, and hold rollout traffic until hit rate stabilized.`",
    "Create a new knowledge record with `record_id` `runbook_cache_warmup_recovery_ap_southeast`, `source_type` `runbook`, `team` `platform`, `region` `ap-southeast`, `product` `developer-platform`, `severity` `sev2`, `release_version` `na`, `last_reviewed_at` `2026-04-09`, title `Runbook: recover cache warmup after regional failover`, and content `If cache warmup lags after failover, restart the warmup workers, verify invalidation backlog drains, pre-warm hot keys, and confirm hit rate recovery before restoring full traffic.`",
    "Replace the existing knowledge record `kb_legacy_invalidation_troubleshooting` with this full record: `record_id` `kb_legacy_invalidation_troubleshooting`, `source_type` `kb_article`, `team` `support`, `region` `global`, `product` `developer-platform`, `severity` `sev2`, `release_version` `na`, `last_reviewed_at` `2026-04-10`, title `KB: troubleshoot stale reads from legacy cache invalidation flow`, content `Support engineers should first confirm whether a service is still publishing to the legacy invalidation path. If so, verify the new event-driven invalidation endpoint is enabled, compare consumer lag, and check for mixed old and new events causing stale objects.`",
    "Replace the existing knowledge record `release_2026_04_new_invalidation_endpoint` with this full record: `record_id` `release_2026_04_new_invalidation_endpoint`, `source_type` `release_note`, `team` `platform`, `region` `global`, `product` `developer-platform`, `severity` `info`, `release_version` `2026.04`, `last_reviewed_at` `2026-04-10`, title `Release notes: new invalidation endpoint is now the default`, content `Release 2026.04 makes the new invalidation endpoint the default for developer-platform services. The legacy cache invalidation flow enters final deprecation on 2026-06-30, and teams should complete migration before that date.`",   
]

In [ ]:
# NBVAL_SKIP
async with agent as client:
    for i, prompt in enumerate(UPSERT_PROMPTS, start=1):
        print("=" * 80)
        print(f"Task {i}: {prompt}\n")
        await client.query(prompt)
        async for msg in client.receive_response():
            print_agent_message(msg)
        print("=" * 80)

Task 1: Create a new knowledge record with `record_id` `incident_ap_southeast_cache_failover_2026_04`, `source_type` `incident_summary`, `team` `platform`, `region` `ap-southeast`, `product` `developer-platform`, `severity` `sev2`, `release_version` `na`, `last_reviewed_at` `2026-04-09`, title `Incident summary: cache warmup lag after ap-southeast failover`, and content `After failover to ap-southeast, cache hit rate dropped because warmup workers started late. The mitigation was to replay invalidation events, pre-warm the highest-traffic keys, and hold rollout traffic until hit rate stabilized.`

Tool use -> mcp__redisvl__upsert-records
{'records': [{'content': 'After failover to ap-southeast, cache hit rate '
                         'dropped because warmup workers started late. The '
                         'mitigation was to replay invalidation events, '
                         'pre-warm the highest-traffic keys, and hold rollout '
                         'traffic until hit rate

## Conclusion

The RedisVL MCP server allows MCP-compatible clients to search or upsert data in an existing Redis index, without the need for explicit coding of these operations. In this notebook, we have built a support agent using Claude Agent SDK and successfully integrated it with RedisVL MCP, using the `search-records` and `upsert-records` tools to perform retrieval and insertion/update.

## Cleanup

In [ ]:
# NBVAL_SKIP
# kill the mcp server process
mcp_server.terminate()

In [28]:
# delete any temp files (the MCP config)
os.remove(MCP_CONFIG_FILEPATH)
os.rmdir("config")